# Statistical Analysis: Chi-square Test and Cramer's V

**Why this notebook is needed:**  
The data are categorical counts in a platform × value-category contingency table. Therefore, the appropriate test is the **chi-square test of independence**, followed by **Cramer's V** as an effect-size measure.

**Research question focus:**  
Do dominant human-value categories differ across Amazon App Store, Apple App Store, and Google Play?


In [3]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Output folder for tables and figures
OUT_DIR = Path("chi_square_rq2_outputs")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be saved to: {OUT_DIR.resolve()}")


Outputs will be saved to: /Users/zhang/2nd_paper experiment/ValueStack_Experiment/chi_square_rq2_outputs


## 1. Input counts from the 3 App Stores(Amazon, Google Play and Apple)

Rows are app-store platforms. Columns are the ten dominant Schwartz value categories. Each cell is the number of annotated reviews assigned to that value category on that platform.


In [4]:
human_values = [
    "Self-direction", "Stimulation", "Hedonism", "Achievement", "Power",
    "Security", "Conformity", "Tradition", "Benevolence", "Universalism"
]

counts = pd.DataFrame(
    {
        "Self-direction": [2776, 1883, 1066],
        "Stimulation":   [1060,  513, 1103],
        "Hedonism":      [1812, 4431, 1370],
        "Achievement":   [ 633,  578,  773],
        "Power":         [ 290, 1380,  397],
        "Security":      [ 833, 1430,  434],
        "Conformity":    [2090,  493, 1800],
        "Tradition":     [  65, 1643,  228],
        "Benevolence":   [1462, 1693, 1200],
        "Universalism":  [ 119,   49,  634],
    },
    index=["Amazon", "Apple App Store", "Google Play"]
)

counts["Platform total"] = counts.sum(axis=1)
counts.loc["Category total"] = counts.sum(axis=0)

display(counts)

# Save counts
counts.to_csv(OUT_DIR / "platform_value_counts.csv")


,Self-direction,Stimulation,Hedonism,Achievement,Power,Security,Conformity,Tradition,Benevolence,Universalism,Platform total
Amazon,2776,1060,1812,633,290,833,2090,65,1462,119,11140
Apple App Store,1883,513,4431,578,1380,1430,493,1643,1693,49,14093
Google Play,1066,1103,1370,773,397,434,1800,228,1200,634,9005
Category total,5725,2676,7613,1984,2067,2697,4383,1936,4355,802,34238


## 3. Chi-square test of independence

Because both variables are categorical, we use the chi-square test of independence.

- **Rows:** app-store platform  
- **Columns:** dominant Schwartz value category  
- **Cells:** number of annotated reviews  

**Null hypothesis:** platform and value category are independent.  
**Alternative hypothesis:** platform and value category are associated.


In [6]:
# Remove totals before running chi-square
observed = counts.drop(index="Category total").drop(columns="Platform total")

chi2, p_value, dof, expected = chi2_contingency(observed, correction=False)

expected_df = pd.DataFrame(
    expected,
    index=observed.index,
    columns=observed.columns
)

print("Chi-square test of independence")
print("--------------------------------")
print(f"chi-square statistic: {chi2:.2f}")
print(f"degrees of freedom:   {dof}")
print(f"p-value:              {p_value:.6g}")

if p_value < 0.001:
    print("Interpretation: p < 0.001, so the association is statistically significant.")
elif p_value < 0.05:
    print("Interpretation: p < 0.05, so the association is statistically significant.")
else:
    print("Interpretation: p >= 0.05, so no statistically significant association is detected.")


Chi-square test of independence
--------------------------------
chi-square statistic: 7507.36
degrees of freedom:   18
p-value:              0
Interpretation: p < 0.001, so the association is statistically significant.


## 4. Cramer's V effect size

Cramer's V measures the strength of association for a contingency table.

\[
V = \sqrt{\frac{\chi^2}{N(k - 1)}}
\]

where \(N\) is the total number of observations and \(k\) is the smaller number of rows or columns.


In [7]:
def cramers_v(chi2_stat, table):
    n = table.to_numpy().sum()
    r, c = table.shape
    k = min(r, c)
    return np.sqrt(chi2_stat / (n * (k - 1)))

v = cramers_v(chi2, observed)

print("Cramer's V")
print("----------")
print(f"N:           {observed.to_numpy().sum():,}")
print(f"Cramer's V:  {v:.3f}")

if v < 0.10:
    strength = "very weak"
elif v < 0.30:
    strength = "weak"
elif v < 0.50:
    strength = "moderate"
else:
    strength = "strong"

print(f"Interpretation: {strength} association.")


Cramer's V
----------
N:           34,238
Cramer's V:  0.331
Interpretation: moderate association.


## 6. Platform-wise percentages and dominant categories

Percentages help explain which categories dominate within each platform.


In [11]:
row_percentages = observed.div(observed.sum(axis=1), axis=0) * 100
row_percentages_rounded = row_percentages.round(2)

display(row_percentages_rounded)

dominant = pd.DataFrame({
    "Dominant value": row_percentages.idxmax(axis=1),
    "Percentage": row_percentages.max(axis=1).round(2),
    "Count": observed.lookup(observed.index, row_percentages.idxmax(axis=1)) if hasattr(observed, "lookup") else [
        observed.loc[idx, row_percentages.idxmax(axis=1).loc[idx]] for idx in observed.index
    ]
})

display(dominant)

row_percentages_rounded.to_csv(OUT_DIR / "platform_value_percentages.csv")
dominant.to_csv(OUT_DIR / "dominant_value_by_platform.csv")


,Self-direction,Stimulation,Hedonism,Achievement,Power,Security,Conformity,Tradition,Benevolence,Universalism
Amazon,24.92,9.52,16.27,5.68,2.60,7.48,18.76,0.58,13.12,1.07
Apple App Store,13.36,3.64,31.44,4.10,9.79,10.15,3.50,11.66,12.01,0.35
Google Play,11.84,12.25,15.21,8.58,4.41,4.82,19.99,2.53,13.33,7.04


,Dominant value,Percentage,Count
Amazon,Self-direction,24.92,2776
Apple App Store,Hedonism,31.44,4431
Google Play,Conformity,19.99,1800


## 7. Expected frequencies

Expected frequencies show what the table would look like if platform and value category were independent.


In [12]:
display(expected_df.round(2))
expected_df.round(2).to_csv(OUT_DIR / "expected_frequencies.csv")


,Self-direction,Stimulation,Hedonism,Achievement,Power,Security,Conformity,Tradition,Benevolence,Universalism
Amazon,1862.74,870.69,2477.04,645.53,672.54,877.52,1426.09,629.92,1416.98,260.95
Apple App Store,2356.52,1101.49,3133.65,816.65,850.82,1110.14,1804.12,796.89,1792.60,330.12
Google Play,1505.74,703.82,2002.31,521.82,543.65,709.34,1152.78,509.19,1145.42,210.94
